# Challenge 9 — Machine Learning Arena

**Nombre:** Miriam Ivonne Castañeda Chavarría
**Clase:** Introducción a Machine Learning  

## Objetivo

Construir y comparar modelos para estimar automáticamente el **precio de una vivienda**.

El dataset utilizado es `viviendas_reto(1).csv`.

## 1. Auditoría del dataset

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/ivonnecastaneda/Downloads/viviendas_reto.csv")

print("Dimensiones:", df.shape)
print("\nTipos:")
print(df.dtypes)
print("\nValores faltantes:")
print(df.isna().sum())
print("\nDuplicados:", df.duplicated().sum())

df.describe()

Dimensiones: (800, 6)

Tipos:
metros              float64
habitaciones          int64
banios              float64
antiguedad            int64
distancia_centro    float64
precio                int64
dtype: object

Valores faltantes:
metros              0
habitaciones        0
banios              6
antiguedad          0
distancia_centro    0
precio              0
dtype: int64

Duplicados: 0


,metros,habitaciones,banios,antiguedad,distancia_centro,precio
count,800.000000,800.000000,794.000000,800.00000,800.000000,8.000000e+02
mean,114.775375,2.966250,2.042821,14.81875,6.866750,2.751383e+06
std,44.385040,1.063732,0.937212,9.67253,6.861149,8.322200e+05
min,30.000000,1.000000,1.000000,0.00000,0.300000,8.020000e+05
25%,82.525000,2.000000,1.000000,8.00000,1.975000,2.170000e+06
50%,113.100000,3.000000,2.000000,13.00000,4.700000,2.768500e+06
75%,146.650000,4.000000,3.000000,20.00000,9.600000,3.277500e+06
max,247.100000,5.000000,4.000000,50.00000,35.000000,8.696600e+06


El objetivo es predecir `precio`.

Las variables disponibles son `metros`, `habitaciones`, `banios`, `antiguedad` y `distancia_centro`. Hay algunos valores faltantes en `banios`, por lo que se imputarán con la mediana dentro de los pipelines.


## 2. Definir X e y

In [4]:
X = df.drop(columns="precio")
y = df["precio"]

print("Variables predictoras:")
print(X.columns.tolist())
print("\nVariable objetivo: precio")

Variables predictoras:
['metros', 'habitaciones', 'banios', 'antiguedad', 'distancia_centro']

Variable objetivo: precio


Las variables se utilizan porque describen características de la vivienda que pueden tener relación con su precio. No se utiliza `precio` como variable de entrada.

## 3. Train / Test

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (640, 5)
Test: (160, 5)


## 4. Baseline

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

baseline = Pipeline([("imputer", SimpleImputer(strategy="median")),("model", DummyRegressor(strategy="mean"))])
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)
mae_baseline = mean_absolute_error(y_test, pred_baseline)

print("MAE baseline:", mae_baseline)

MAE baseline: 683370.046875


El baseline sirve como referencia. Los modelos reales deberían mejorar este resultado para demostrar que las variables aportan información.

## 5. Torneo de modelos

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

modelos = {"Baseline": Pipeline([("imputer", SimpleImputer(strategy="median")),("model", DummyRegressor(strategy="mean"))]),
    "Lineal": Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler()),("model", LinearRegression())]),
    "KNN": Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler()),("model", KNeighborsRegressor(n_neighbors=5))]),
    "Árbol": Pipeline([("imputer", SimpleImputer(strategy="median")),("model", DecisionTreeRegressor(random_state=42))]),
    "Random Forest": Pipeline([("imputer", SimpleImputer(strategy="median")),("model", RandomForestRegressor(n_estimators=100,random_state=42))])}

resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    resultados.append([nombre, mae, rmse, r2])

resultados = pd.DataFrame(resultados,columns=["Modelo", "MAE", "RMSE", "R2"])

resultados.sort_values("MAE")

,Modelo,MAE,RMSE,R2
1,Lineal,179049.904653,484645.553992,0.730501
4,Random Forest,216383.700000,520927.010759,0.688641
2,KNN,272553.000000,551028.188117,0.651618
3,Árbol,312302.500000,638464.157960,0.532285
0,Baseline,683370.046875,936603.674341,-0.006514


## 6. Generalización del modelo ganador

In [11]:
# En este caso tomamos el modelo con menor MAE,
# sin contar el baseline.

modelo_ganador = resultados[resultados["Modelo"] != "Baseline"].sort_values("MAE").iloc[0]["Modelo"]
print("Modelo ganador por MAE:", modelo_ganador)
modelo = modelos[modelo_ganador]
pred_train = modelo.predict(X_train)
pred_test = modelo.predict(X_test)
mae_train = mean_absolute_error(y_train, pred_train)
mae_test = mean_absolute_error(y_test, pred_test)

print("MAE train:", mae_train)
print("MAE test:", mae_test)
print("Brecha:", mae_test - mae_train)

Modelo ganador por MAE: Lineal
MAE train: 135777.128017506
MAE test: 179049.90465316692
Brecha: 43272.776635660906


La diferencia entre MAE de train y test ayuda a detectar posibles problemas de overfitting. Una brecha grande significa que el modelo funciona mucho mejor con los datos que ya vio que con datos nuevos.

## 7. Hiperparámetros

In [12]:
# Probamos Random Forest con más árboles

rf_300 = Pipeline([("imputer", SimpleImputer(strategy="median")),("model", RandomForestRegressor(n_estimators=300,random_state=42))])
rf_300.fit(X_train, y_train)
pred_rf300 = rf_300.predict(X_test)
mae_rf300 = mean_absolute_error(y_test, pred_rf300)
rmse_rf300 = np.sqrt(mean_squared_error(y_test, pred_rf300))
r2_rf300 = r2_score(y_test, pred_rf300)

print("Random Forest 300 árboles")
print("MAE:", mae_rf300)
print("RMSE:", rmse_rf300)
print("R2:", r2_rf300)

Random Forest 300 árboles
MAE: 217384.69166666665
RMSE: 521015.92246543785
R2: 0.6885344697501852


Se modifica `n_estimators` de 100 a 300 para revisar si aumentar el número de árboles cambia el desempeño.

## 8. Cinco errores más grandes

In [13]:
pred = modelo.predict(X_test)

errores = pd.DataFrame({"precio_real": y_test.values,"prediccion": pred,"error_absoluto": np.abs(y_test.values - pred)}, index=X_test.index)
top_5 = errores.sort_values("error_absoluto",ascending=False).head(5)
top_5

,precio_real,prediccion,error_absoluto
545,8696600,3.889527e+06,4.807073e+06
360,6028000,2.814933e+06,3.213067e+06
65,1224000,1.634583e+06,4.105834e+05
446,2343000,1.971289e+06,3.717114e+05
316,2017000,2.386088e+06,3.690880e+05


También revisamos las características de esas viviendas para buscar algún patrón.

In [15]:
detalle_errores = X_test.loc[top_5.index].copy()
detalle_errores["precio_real"] = y_test.loc[top_5.index]
detalle_errores["prediccion"] = top_5["prediccion"]
detalle_errores["error_absoluto"] = top_5["error_absoluto"]
detalle_errores

,metros,habitaciones,banios,antiguedad,distancia_centro,precio_real,prediccion,error_absoluto
545,167.1,4,3.0,19,4.1,8696600,3.889527e+06,4.807073e+06
360,103.9,3,4.0,10,0.6,6028000,2.814933e+06,3.213067e+06
65,47.1,3,2.0,19,0.5,1224000,1.634583e+06,4.105834e+05
446,65.6,3,3.0,6,10.6,2343000,1.971289e+06,3.717114e+05
316,97.5,3,3.0,12,21.7,2017000,2.386088e+06,3.690880e+05


# Conclusión ejecutiva

Se compararon un baseline y cuatro modelos de regresión para estimar el precio de viviendas.  
El modelo ganador se determina utilizando el MAE obtenido en el conjunto de prueba.  
También se comparó el error de entrenamiento contra el error de prueba para revisar posibles señales de overfitting.  
Después se modificó un hiperparámetro de Random Forest para observar si el desempeño cambiaba.  
Finalmente, se revisaron las cinco viviendas con mayor error absoluto para identificar dónde el modelo tiene más dificultad.  
Los resultados deben interpretarse con las métricas obtenidas y no solamente con un único indicador.
